# 12 Attention 反向传播逐步推导

上一课建立了 Attention 的梯度地图。

这一课沿着地图逐段计算：

$$
O=AV\rightarrow A=\operatorname{softmax}(S)\rightarrow S=\frac{QK^{\top}}{\sqrt{d_k}}\rightarrow(Q,K,V)=(XW_Q,XW_K,XW_V)
$$

目标不是死记所有公式，而是理解每个公式为什么具有这样的形状和含义。

## 1. 统一记号

为了让公式更短，我们把某个变量 $Z$ 收到的梯度记为：

$$
G_Z=\frac{\partial\mathcal{L}}{\partial Z}
$$

例如：

$$
G_O=\frac{\partial\mathcal{L}}{\partial O}
$$

$G_O$ 可以理解为后续网络传回 Attention 输出的“修改意见”。

它的具体数值由损失函数和后续网络决定。本课从已经得到 $G_O$ 开始。

## 2. 第一步：从 $O=AV$ 传回 $A$ 和 $V$

前向传播是：

$$
O=AV
$$

假设：

$$
A:N\times N,\qquad V:N\times d_v,\qquad O:N\times d_v.
$$

反向传播得到：

$$
\boxed{G_A=G_OV^{\top}}
$$

$$
\boxed{G_V=A^{\top}G_O}
$$

## 3. 为什么 $G_A=G_OV^{\top}$

第 $i$ 个输出 token 是：

$$
o_i=\sum_{j=1}^{N}a_{ij}v_j
$$

如果某个 $a_{ij}$ 稍微变化，它影响输出的方向由 $v_j$ 决定。

所以要判断 $a_{ij}$ 应该增大还是减小，需要比较：

- 输出收到的修改意见 $G_{O,i}$；
- 第 $j$ 个 Value 的方向 $v_j$。

两者做点积，正好组成矩阵乘法 $G_OV^{\top}$。

形状也能对上：

$$
(N\times d_v)\cdot(d_v\times N)\rightarrow N\times N.
$$

## 4. 为什么 $G_V=A^{\top}G_O$

同一个 Value $v_j$ 可能被多个 Query 使用。

因此 $v_j$ 收到的梯度，要汇总所有输出位置传回来的修改意见，并按照当时使用它的注意力权重加权。

矩阵形式就是：

$$
G_V=A^{\top}G_O.
$$

形状为：

$$
(N\times N)\cdot(N\times d_v)\rightarrow N\times d_v.
$$

这一步可以理解为：前向传播用 $A$ 把 Value 送到各个输出位置；反向传播用 $A^{\top}$ 把各个输出位置的梯度送回 Value。

## 5. 第二步：梯度怎样穿过 Softmax

对注意力表的第 $i$ 行：

$$
a_i=\operatorname{softmax}(s_i).
$$

Softmax 的每个输出不只依赖一个输入，而是依赖这一整行的所有分数。其局部导数是：

$$
\frac{\partial a_{ij}}{\partial s_{ik}}=a_{ij}(\delta_{jk}-a_{ik}),
$$

其中 $\delta_{jk}$ 在 $j=k$ 时为 1，否则为 0。

对整行写成便于计算的形式：

$$
\boxed{G_{S,i}=a_i\odot\left(G_{A,i}-\left(G_{A,i}^{\top}a_i\right)\mathbf{1}\right)}
$$

$\odot$ 表示逐元素相乘。

## 6. Softmax 梯度的直觉：同一行内存在竞争

一行 Softmax 权重之和永远是 1。

因此提高一个位置的权重，通常会挤压同一行其他位置的权重。Softmax 的梯度不能把每个格子完全独立处理。

公式中的：

$$
G_{A,i}^{\top}a_i
$$

可以理解为这一行修改意见的加权平均。每个位置先减去这个共同基准，再乘上自己的当前权重。

还有一个重要现象：

$$
\sum_{j=1}^{N}(G_S)_{ij}=0.
$$

因为给一整行所有分数同时加上相同常数，不会改变 Softmax 的结果。

## 7. 第三步：从分数 $S$ 传回 $Q$ 和 $K$

前向传播是：

$$
S=\frac{QK^{\top}}{\sqrt{d_k}}.
$$

反向传播得到：

$$
\boxed{G_Q=\frac{G_SK}{\sqrt{d_k}}}
$$

$$
\boxed{G_K=\frac{G_S^{\top}Q}{\sqrt{d_k}}}
$$

形状检查：

$$
\begin{aligned}
G_Q &: (N\times N)\cdot(N\times d_k)\rightarrow N\times d_k,\\
G_K &: (N\times N)\cdot(N\times d_k)\rightarrow N\times d_k.
\end{aligned}
$$

## 8. 为什么 $K$ 的公式里有转置

分数矩阵中的一个格子是：

$$
s_{ij}=\frac{q_i\cdot k_j}{\sqrt{d_k}}.
$$

一行表示某个 Query 看所有 Key，一列表示某个 Key 被所有 Query 看。

计算某个 $k_j$ 的梯度时，需要汇总分数表第 $j$ 列的梯度。因此矩阵计算要把 $G_S$ 转置：

$$
G_K=\frac{G_S^{\top}Q}{\sqrt{d_k}}.
$$

这与 $G_V=A^{\top}G_O$ 的直觉相似：一个 Key 或 Value 被多个输出位置使用，所以反向时需要按列汇总。

## 9. 第四步：传回 $W_Q$、$W_K$、$W_V$

对于三个线性变换：

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V,
$$

参数梯度分别是：

$$
\boxed{G_{W_Q}=X^{\top}G_Q}
$$

$$
\boxed{G_{W_K}=X^{\top}G_K}
$$

$$
\boxed{G_{W_V}=X^{\top}G_V}
$$

如果 $X:N\times D$，那么：

$$
\begin{aligned}
G_{W_Q},G_{W_K}&:D\times d_k,\\
G_{W_V}&:D\times d_v.
\end{aligned}
$$

## 10. 第五步：三条路线汇总到输入 $X$

每个线性分支传给输入的梯度分别是：

$$
\begin{aligned}
G_X^{(Q)}&=G_QW_Q^{\top},\\
G_X^{(K)}&=G_KW_K^{\top},\\
G_X^{(V)}&=G_VW_V^{\top}.
\end{aligned}
$$

总梯度为：

$$
\boxed{G_X=G_QW_Q^{\top}+G_KW_K^{\top}+G_VW_V^{\top}}
$$

如果 $X$ 是上一层 Transformer 的输出，这个 $G_X$ 还会继续向更前面的网络传播。

## 11. 把完整反向传播公式串起来

已知上游梯度 $G_O$，单头 Attention 的核心路线是：

$$
\begin{aligned}
G_A&=G_OV^{\top}, & G_V&=A^{\top}G_O,\\
G_{S,i}&=a_i\odot\left(G_{A,i}-(G_{A,i}^{\top}a_i)\mathbf{1}\right),\\
G_Q&=\frac{G_SK}{\sqrt{d_k}}, & G_K&=\frac{G_S^{\top}Q}{\sqrt{d_k}},\\
G_{W_Q}&=X^{\top}G_Q, & G_{W_K}&=X^{\top}G_K, & G_{W_V}&=X^{\top}G_V,\\
G_X&=G_QW_Q^{\top}+G_KW_K^{\top}+G_VW_V^{\top}.
\end{aligned}
$$

现在看起来公式很多，但它们都来自四种已经见过的操作：矩阵乘法、转置、Softmax 和分支梯度相加。

## 12. 多头 Attention 会不会换一套反向传播

不会。

多头 Attention 只是让多个 head 分别执行同样的计算，再把结果拼接并经过输出投影。

每个 head 内部仍然使用本课的反向传播路线：

$$
O_r=A_rV_r\rightarrow A_r=\operatorname{softmax}(S_r)\rightarrow S_r=\frac{Q_rK_r^{\top}}{\sqrt{d_{\mathrm{head}}}}.
$$

拼接操作在反向传播时，会把梯度重新切分给各个 head。各个 head 再独立把梯度传给自己的 QKV 参数。

## 13. 这一课真正需要记住什么

不要求现在默写 Softmax 的完整导数。优先掌握下面四点：

1. 矩阵乘法的反向传播会把梯度分别传给左右两个输入。
2. Softmax 的一行内部相互关联，不能把每个权重完全独立求导。
3. $QK^{\top}$ 让梯度同时传给 Q 和 K。
4. 同一个 $X$ 走了 Q、K、V 三条分支，所以返回时要把三条梯度相加。

下一课将用 PyTorch 实际观察这些梯度，并让自动求导结果和本课公式互相验证。

## 14. 自测问题

1. 已知 $G_O$，怎样计算 $G_A$ 和 $G_V$？
2. 为什么 $G_A$ 的形状是 $N\times N$？
3. 为什么 Softmax 的某个输出会受到同一行其他分数的影响？
4. 怎样从 $G_S$ 计算 $G_Q$ 和 $G_K$？
5. $1/\sqrt{d_k}$ 在反向传播中是否消失？
6. 怎样计算 $W_Q$、$W_K$、$W_V$ 的梯度？
7. 为什么 $G_X$ 是三项之和？
8. 多头 Attention 是否需要一套完全不同的反向传播原理？